In [ ]:
import torch # 파이토치 기본 라이브러리 (텐서 연산 및 그래프 계산)
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. GPU 설정 (Colab에서 매우 중요)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 데이터 전처리 및 로드
# LeNet-5의 원본 입력은 32x32입니다. MNIST(28x28)를 32x32로 키워줍니다.
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST 평균, 표준편차
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# 3. LeNet-5 모델 정의
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # C1: Convolutional Layer (입력 1, 출력 6, 커널 5x5)
        self.c1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1)
        # S2: Average Pooling Layer (2x2 커널, 스트라이드 2)
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
        # C3: Convolutional Layer (입력 6, 출력 16, 커널 5x5)
        self.c3 = nn.Conv2d(6, 16, 5)
        # S4: Average Pooling Layer (2x2 커널, 스트라이드 2)
        self.s4 = nn.AvgPool2d(2, 2)
        # C5: Fully Connected Layer (16x5x5 -> 120)
        self.c5 = nn.Linear(16 * 5 * 5, 120)
        # F6: Fully Connected Layer (120 -> 84)
        self.f6 = nn.Linear(120, 84)
        # Output: Fully Connected Layer (84 -> 10)
        self.output = nn.Linear(84, 10)

        self.tanh = nn.Tanh() # 원본 논문에서는 Tanh 활성화 함수 사용

    def forward(self, x):
        x = self.tanh(self.c1(x))
        x = self.s2(x)
        x = self.tanh(self.c3(x))
        x = self.s4(x)

        # Flatten: 텐서를 1차원으로 펼침 (배치 크기는 유지)
        x = x.view(-1, 16 * 5 * 5)

        x = self.tanh(self.c5(x))
        x = self.tanh(self.f6(x))
        x = self.output(x)
        return x

# 4. 모델, 손실함수, 최적화 도구 선언
model = LeNet5().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. 학습 루프 (생략 가능하지만 흐름 이해를 위해 간단히 작성)
for epoch in range(5):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/5], Loss: {loss.item():.4f}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 407kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.75MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.32MB/s]


Epoch [1/5], Loss: 0.1432
Epoch [2/5], Loss: 0.0792
Epoch [3/5], Loss: 0.0044
Epoch [4/5], Loss: 0.0125
Epoch [5/5], Loss: 0.0049
